## 1. Import library (Ketua)
Semua import dikumpulkan di satu sel ini supaya gampang dicek. Yang dipakai cuma library, tidak ada import antar-file.

In [ ]:
import math
import random
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

## 2. Config (Ketua)
Semua angka dan path proyek disimpan di sini, jadi sel lain cukup membaca `config.X` tanpa menulis angka sendiri. Isinya dibagi tiga sel (root repo, nilai baseline, lalu sisanya) dan harus dijalankan berurutan dari atas. `BATCH_SIZE` dan `EPOCHS` masih sementara sampai uji memori di GTX 1650 Ti.

In [ ]:
# Root repo: folder kerja saat ini, atau folder induknya kalau dijalankan dari notebooks/
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

In [ ]:
config = SimpleNamespace(
    # Reproducibility
    SEED=42,  # dipakai set_seed(); satu seed untuk semua anggota
    # Training (baseline)
    LEARNING_RATE=2e-4,  # baseline; boleh ditimpa konfigurasi di EXPERIMENTS
    # SEMENTARA, BELUM FINAL: menunggu uji memori di GTX 1650 Ti (4GB).
    # Jangan dipakai untuk eksperimen tuning sebelum nilai final ditetapkan.
    BATCH_SIZE=64,
    EPOCHS=30,
    NUM_WORKERS=0,  # DataLoader tanpa proses worker (aman di macOS dan Windows)
    # Diffusion (baseline)
    T=1000,  # jumlah timestep baseline; boleh ditimpa konfigurasi di EXPERIMENTS
    # Acuan untuk T=1000; Diffusion menskalakan beta dengan BETA_REFERENCE_T / T
    BETA_START=1e-4,
    BETA_END=0.02,
    BETA_REFERENCE_T=1000,
    # Model
    IMAGE_SIZE=28,  # sisi gambar MNIST; channel ada di MODEL_PARAMS
    # Dipakai sebagai UNet(**config.MODEL_PARAMS) dan disimpan di checkpoint.
    # channel_mults (1, 2, 2) = dua downsampling: 28 -> 14 -> 7
    MODEL_PARAMS={
        "in_channels": 1,
        "base_channels": 64,
        "channel_mults": (1, 2, 2),
        "time_emb_dim": 128,
    },
)

In [ ]:
# Sampling selama training
config.SAMPLE_EVERY_N_EPOCHS = 5  # grid sampel disimpan tiap N epoch
config.NUM_SAMPLE_IMAGES = 16     # jumlah gambar per grid

# Smoke test (uji alur cepat, bukan untuk hasil)
config.SMOKE_TEST = False         # True = subset kecil dan epoch pendek
config.SMOKE_SUBSET_SIZE = 512    # jumlah gambar pada mode smoke test
config.SMOKE_EPOCHS = 2           # jumlah epoch pada mode smoke test
config.SMOKE_SAMPLE_EVERY = 1     # interval grid smoke test (harus <= SMOKE_EPOCHS)

# Path: semuanya diturunkan dari BASE_DIR (sel sebelumnya)
config.BASE_DIR = BASE_DIR
config.DATA_DIR = BASE_DIR / "data"               # unduhan MNIST (tidak di-commit)
config.CHECKPOINT_DIR = BASE_DIR / "checkpoints"  # checkpoint (tidak di-commit)
config.RESULTS_DIR = BASE_DIR / "results"         # hasil (di-commit)
config.LOSS_CURVES_DIR = config.RESULTS_DIR / "loss_curves"
config.SAMPLES_DIR = config.RESULTS_DIR / "samples"

In [ ]:
# Uji config: konsistensi nilai dan path. Tidak membuat folder.
print(f"BASE_DIR       : {config.BASE_DIR}")
for name in ("DATA_DIR", "CHECKPOINT_DIR", "RESULTS_DIR", "LOSS_CURVES_DIR", "SAMPLES_DIR"):
    path = getattr(config, name)
    assert path.is_absolute(), f"{name} harus absolut"
    assert config.BASE_DIR in path.parents, f"{name} harus berada di bawah BASE_DIR"
    print(f"{name:<15}: {path}")

assert config.BASE_DIR.name != "notebooks", "BASE_DIR harus root repo, bukan folder notebooks"

assert 0 < config.BETA_START < config.BETA_END < 1, "beta harus 0 < start < end < 1"
assert config.T > 0 and config.BATCH_SIZE > 0 and config.EPOCHS > 0
assert config.BETA_REFERENCE_T > 0 and config.IMAGE_SIZE > 0
assert config.BETA_END * config.BETA_REFERENCE_T / config.T < 1, "beta akhir (setelah skala) harus < 1"
assert config.SMOKE_SAMPLE_EVERY <= config.SMOKE_EPOCHS, "smoke: N harus <= jumlah epoch"
assert config.SMOKE_SUBSET_SIZE >= config.BATCH_SIZE, "subset smoke harus >= batch size"

# Dua downsampling dari IMAGE_SIZE: 28 -> 14 -> 7
n_down = len(config.MODEL_PARAMS["channel_mults"]) - 1
size = config.IMAGE_SIZE
for _ in range(n_down):
    assert size % 2 == 0, "ukuran gambar harus habis dibagi 2 di tiap downsampling"
    size //= 2
assert n_down == 2 and size == 7, "U-Net harus 28 -> 14 -> 7"

print(f"SEED={config.SEED}  LR={config.LEARNING_RATE}  T={config.T}  "
      f"BATCH_SIZE={config.BATCH_SIZE}(sementara)  EPOCHS={config.EPOCHS}(sementara)")
print(f"BETA_REFERENCE_T={config.BETA_REFERENCE_T}  IMAGE_SIZE={config.IMAGE_SIZE}")
print(f"MODEL_PARAMS   : {config.MODEL_PARAMS}")
print("config OK")

## 3. Utils (Ketua)
Dua fungsi kecil yang dipakai semua bagian: `set_seed` supaya hasil acak bisa diulang, dan `get_device` untuk memilih cuda, mps, atau cpu. Device hanya ditentukan di `get_device`, jadi tidak ada sel lain yang menulis "cuda" atau "cpu" langsung.

In [ ]:
def set_seed(seed):
    """Seed random, numpy, torch. Tanpa mode deterministik penuh agar training tidak melambat."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # sekaligus menyemai CUDA dan MPS

In [ ]:
def get_device():
    """Device pertama yang tersedia: cuda, mps, cpu. Tanpa print."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [ ]:
# Uji set_seed: seed sama -> angka sama, seed beda -> angka beda
def draw():
    return (random.random(), float(np.random.rand()), float(torch.rand(1)))

set_seed(42)
a = draw()
set_seed(42)
b = draw()
set_seed(7)
c = draw()
assert a == b, "seed sama harus menghasilkan angka sama"
assert a != c, "seed beda harus menghasilkan angka beda"
print("set_seed OK: reproducible untuk random, numpy, torch")

# Uji get_device: tipe benar, urutan prioritas, bisa dipakai
device = get_device()
assert isinstance(device, torch.device)
assert device.type in ("cuda", "mps", "cpu")

if torch.cuda.is_available():
    expected = "cuda"
elif torch.backends.mps.is_available():
    expected = "mps"
else:
    expected = "cpu"
assert device.type == expected, "urutan prioritas harus cuda > mps > cpu"

x = torch.ones(2, 2, device=device)
assert float((x + x).sum().cpu()) == 8.0
print(f"get_device OK: {device}")

## 6. Model (Ketua)
Di bagian ini kita bikin otak DDPM-nya: timestep embedding, blok dasar, dan U-Net yang menebak noise di gambar.

In [ ]:
def _group_norm(channels):
    # Jumlah grup terbesar dari (8, 4, 2, 1) yang habis membagi jumlah channel
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return nn.GroupNorm(groups, channels)

Angka `t` mentah susah dipakai jaringan, jadi diubah jadi vektor berisi gelombang sin dan cos dengan frekuensi berbeda-beda, mirip jarum jam, menit, dan detik yang bareng-bareng menunjuk satu waktu. Model perlu tahu `t` karena kerjanya beda: di `t` kecil cuma merapikan detail, di `t` besar harus menebak bentuk digit dari noise.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """t (B,) bertipe long -> vektor (B, dim) berisi sin dan cos berbagai frekuensi."""

    def __init__(self, dim):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("time_emb_dim harus genap")
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        # Frekuensi turun geometris dari 1 ke 1/10000
        exponent = -math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half
        freqs = torch.exp(exponent)                      # (half,)
        args = t.float()[:, None] * freqs[None, :]       # (B, half)
        return torch.cat([args.sin(), args.cos()], dim=1)  # (B, dim)

`ResBlock` itu dua konvolusi 3x3 yang di tengahnya ditambah embedding waktu, supaya tiap blok tahu lagi di `t` berapa. Inputnya juga dijumlahkan ke keluaran (jalan pintas residual) biar training lebih stabil.

In [ ]:
class ResBlock(nn.Module):
    """Dua konvolusi 3x3; embedding waktu ditambahkan di antaranya."""

    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.norm1 = _group_norm(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch))
        self.norm2 = _group_norm(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act = nn.SiLU()
        # Jalan pintas residual; 1x1 conv bila jumlah channel berubah
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]  # (B,C) -> (B,C,1,1), disiarkan ke semua piksel
        h = self.conv2(self.act(self.norm2(h)))
        return h + self.shortcut(x)

`Downsample` memperkecil gambar jadi setengahnya (28 → 14 → 7) pakai konvolusi stride 2. `Upsample` kebalikannya: ukuran gambar digandakan (7 → 14 → 28), lalu dihaluskan dengan satu konvolusi.

In [ ]:
class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)  # 28->14, 14->7

    def forward(self, x):
        return self.conv(x)

In [ ]:
class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")  # 7->14, 14->28
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        return self.conv(self.up(x))

`UNet` punya jalur turun (28 → 14 → 7), satu bagian tengah di 7x7, lalu jalur naik (7 → 14 → 28). Hasil tiap level di jalur turun disimpan, lalu ditempel (skip connection) ke level naik yang ukurannya sama, supaya detail halus yang hilang waktu diperkecil tetap kebawa. Keluarannya prediksi noise berukuran sama dengan input, yaitu (B, 1, 28, 28).

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels, base_channels, channel_mults, time_emb_dim):
        super().__init__()
        n = len(channel_mults)
        if not 1 <= n <= 3:
            raise ValueError("channel_mults harus berisi 1 sampai 3 level (maks. dua downsampling)")
        chs = [base_channels * m for m in channel_mults]
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.init_conv = nn.Conv2d(in_channels, chs[0], 3, padding=1)
        self.down_blocks = nn.ModuleList(
            [ResBlock(chs[max(i - 1, 0)], chs[i], time_emb_dim) for i in range(n)]
        )
        self.downsamples = nn.ModuleList([Downsample(chs[i]) for i in range(n - 1)])
        self.mid1 = ResBlock(chs[-1], chs[-1], time_emb_dim)
        self.mid2 = ResBlock(chs[-1], chs[-1], time_emb_dim)

        # Jalur naik dari level terdalam: input = hasil level bawah + skip
        self.up_blocks = nn.ModuleList()
        for k in range(n):
            i = n - 1 - k
            self.up_blocks.append(
                ResBlock(chs[i] + chs[min(i + 1, n - 1)], chs[i], time_emb_dim)
            )
        self.upsamples = nn.ModuleList([Upsample(chs[n - 1 - k]) for k in range(n - 1)])
        self.out_norm = _group_norm(chs[0])
        self.out_conv = nn.Conv2d(chs[0], in_channels, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        h = self.init_conv(x)
        skips = []
        for i, block in enumerate(self.down_blocks):
            h = block(h, t_emb)
            skips.append(h)
            if i < len(self.downsamples):
                h = self.downsamples[i](h)
        h = self.mid2(self.mid1(h, t_emb), t_emb)

        for k, block in enumerate(self.up_blocks):
            h = torch.cat([h, skips.pop()], dim=1)  # skip dari level seukuran
            h = block(h, t_emb)
            if k < len(self.upsamples):
                h = self.upsamples[k](h)
        return self.out_conv(self.act(self.out_norm(h)))

Tiga hal yang dicek: shape keluaran harus sama dengan input, `t` yang beda harus ngasih keluaran yang beda, dan satu langkah backward harus jalan buat semua parameter.

In [ ]:
set_seed(config.SEED)
device = get_device()
model = UNet(**config.MODEL_PARAMS).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"device: {device} | jumlah parameter: {n_params:,}")

B, C, S = 4, config.MODEL_PARAMS["in_channels"], config.IMAGE_SIZE
x = torch.randn(B, C, S, S, device=device)
t = torch.randint(0, config.T, (B,), device=device)  # long, 0..T-1
out = model(x, t)
assert out.shape == x.shape, f"shape salah: {tuple(out.shape)}"
assert torch.isfinite(out).all()
print(f"shape OK: {tuple(x.shape)} -> {tuple(out.shape)}")

In [ ]:
with torch.no_grad():
    o0 = model(x, torch.zeros(B, dtype=torch.long, device=device))
    o1 = model(x, torch.full((B,), config.T - 1, dtype=torch.long, device=device))
assert (o0 - o1).abs().max().item() > 0
print("pengaruh t OK")

In [ ]:
loss = nn.functional.mse_loss(model(x, t), torch.randn_like(x))
loss.backward()
missing = [n for n, p in model.named_parameters() if p.grad is None]
assert not missing, f"tanpa gradien: {missing}"
assert all(torch.isfinite(p.grad).all() for p in model.parameters())
print(f"backward OK, loss={loss.item():.4f}")
print("model OK")

## 7. Diffusion (Ketua)
Gambar dirusak pelan-pelan jadi noise lewat T langkah kecil: `beta` adalah porsi noise tiap langkah, dan `alpha_bar` adalah sisa gambar asli setelah t langkah (turun dari sekitar 1 ke sekitar 0). Karena itu gambar di langkah t bisa dihitung langsung dari gambar bersih dengan satu rumus. Kalau T diubah, beta dikalikan `BETA_REFERENCE_T / T` supaya langkah terakhir tetap noise murni; sampling lalu membalik prosesnya dari noise murni, langkah T-1 sampai 0.

In [ ]:
def _extract(values, t, x):
    """Ambil values[t] lalu ubah ke (B,1,1,1) agar bisa dikalikan dengan x (B,C,H,W)."""
    out = values.to(x.device)[t.to(x.device)]
    return out.reshape(t.shape[0], *([1] * (x.dim() - 1)))

In [ ]:
class Diffusion:
    def __init__(self, T, device=None,
                 beta_start=config.BETA_START, beta_end=config.BETA_END):
        # Beta diskalakan BETA_REFERENCE_T / T supaya total noise tetap sama saat T berubah
        scale = config.BETA_REFERENCE_T / T
        if beta_end * scale >= 1.0:
            raise ValueError(f"T={T} terlalu kecil: beta akhir jadi >= 1")
        self.T = T
        self.betas = torch.linspace(beta_start * scale, beta_end * scale, T, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_bars = self.alpha_bars.sqrt()
        self.sqrt_one_minus_alpha_bars = (1.0 - self.alpha_bars).sqrt()
        self.sqrt_recip_alphas = (1.0 / self.alphas).sqrt()
        self.sqrt_betas = self.betas.sqrt()  # simpangan baku noise tambahan saat sampling

    def add_noise(self, x0, t, noise):
        # Forward process: x_t = sqrt(a_bar_t) * x0 + sqrt(1 - a_bar_t) * noise
        return (_extract(self.sqrt_alpha_bars, t, x0) * x0
                + _extract(self.sqrt_one_minus_alpha_bars, t, x0) * noise)

    def sample_timesteps(self, batch_size, device):
        return torch.randint(0, self.T, (batch_size,), device=device)  # t acak long, 0..T-1

    @torch.no_grad()
    def sample(self, model, num_images, device):
        # Reverse process; mode eval/train model diatur pemanggil (train)
        size = config.IMAGE_SIZE
        shape = (num_images, config.MODEL_PARAMS["in_channels"], size, size)
        x = torch.randn(shape, device=device)
        for i in reversed(range(self.T)):
            t = torch.full((num_images,), i, device=device, dtype=torch.long)
            pred_noise = model(x, t)
            # Buang noise yang diprediksi -> rata-rata estimasi x_{t-1}
            coef = _extract(self.betas, t, x) / _extract(self.sqrt_one_minus_alpha_bars, t, x)
            mean = _extract(self.sqrt_recip_alphas, t, x) * (x - coef * pred_noise)
            # Noise baru tiap langkah, kecuali langkah terakhir (t=0)
            if i > 0:
                x = mean + _extract(self.sqrt_betas, t, x) * torch.randn_like(x)
            else:
                x = mean
        return x.clamp(-1.0, 1.0)

In [ ]:
# Uji 1: noise schedule
set_seed(config.SEED)
device = get_device()
C = config.MODEL_PARAMS["in_channels"]

ref_beta_sum = Diffusion(config.BETA_REFERENCE_T).betas.sum().item()
for T_test in (100, 500, 1000, 2000):
    d = Diffusion(T_test)
    ab = d.alpha_bars
    assert d.betas.shape == (T_test,) and ab.shape == (T_test,)
    assert ((d.betas > 0) & (d.betas < 1)).all()
    assert (ab[1:] < ab[:-1]).all(), "alpha-bar harus menurun"
    assert ab[0] > 0.99 and ab[-1] < 1e-3
    assert abs(d.betas.sum().item() - ref_beta_sum) < 1e-2, "total beta harus tetap"
    print(f"schedule OK  T={T_test:<5} alpha_bar[0]={ab[0]:.5f}  alpha_bar[-1]={ab[-1]:.2e}")

try:
    Diffusion(10)
    raise AssertionError("T=10 seharusnya ditolak")
except ValueError:
    print("T terlalu kecil ditolak OK")

In [ ]:
# Uji 2: forward process pada gambar sintetis (kotak putih di latar hitam)
d = Diffusion(config.T, device)
x0 = torch.full((8, C, config.IMAGE_SIZE, config.IMAGE_SIZE), -1.0, device=device)
x0[:, :, 8:20, 8:20] = 1.0
noise = torch.randn_like(x0)

errors = []
for t_val in (0, d.T // 4, d.T // 2, d.T - 1):
    t = torch.full((8,), t_val, device=device, dtype=torch.long)
    xt = d.add_noise(x0, t, noise)
    assert xt.shape == x0.shape
    errors.append(((xt - x0) ** 2).mean().item())
assert all(a < b for a, b in zip(errors, errors[1:])), "gambar harus makin berisik"
assert (xt - noise).abs().max().item() < 0.05, "t=T-1 harus ~ noise murni"
print("forward process OK  MSE ke gambar asli:", [f"{e:.4f}" for e in errors])

In [ ]:
# Visualisasi forward process: makin besar t, makin berisik
ts = [0, d.T // 4, d.T // 2, 3 * d.T // 4, d.T - 1]
fig, axes = plt.subplots(1, len(ts), figsize=(2.2 * len(ts), 2.6))
for ax, t_val in zip(axes, ts):
    t = torch.full((1,), t_val, device=device, dtype=torch.long)
    xt = d.add_noise(x0[:1], t, noise[:1])
    ax.imshow(xt[0, 0].cpu().numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"t={t_val}")
    ax.axis("off")
plt.show()

In [ ]:
# Uji 3: bentuk tertutup = menambah noise langkah demi langkah (uji statistik)
t_check = d.T // 3
x = torch.ones(20000, device=device)
for i in range(t_check + 1):
    x = d.alphas[i].sqrt() * x + d.betas[i].sqrt() * torch.randn_like(x)
assert abs(x.mean().item() - d.sqrt_alpha_bars[t_check].item()) < 0.05
assert abs(x.std().item() - d.sqrt_one_minus_alpha_bars[t_check].item()) < 0.05
print("bentuk tertutup = langkah bertahap OK")

In [ ]:
# Uji 4: t acak bertipe long, 0..T-1, kedua ujung tercapai
small = Diffusion(50, device)
ts = small.sample_timesteps(2000, device)
assert ts.shape == (2000,) and ts.dtype == torch.long
assert ts.min().item() == 0 and ts.max().item() == 49
print("sample_timesteps OK")

In [ ]:
# Uji 5: sampling dengan model tiruan (prediksi noise nol) yang mencatat t
class SpyModel:
    def __init__(self):
        self.t_seen = []

    def __call__(self, x, t):
        assert t.shape == (x.shape[0],) and t.dtype == torch.long
        self.t_seen.append(int(t[0]))
        return torch.zeros_like(x)

N = 4
spy = SpyModel()
out = d.sample(spy, N, device)
assert out.shape == (N, C, config.IMAGE_SIZE, config.IMAGE_SIZE)
assert torch.isfinite(out).all() and out.min() >= -1 and out.max() <= 1
assert not out.requires_grad and out.device.type == device.type
assert spy.t_seen == list(range(d.T - 1, -1, -1)), "t harus T-1 turun ke 0"
print("sampling (model tiruan) OK")

In [ ]:
# Uji 6: model oracle (tahu gambar asli) -> hasil sampling harus kembali ke gambar itu
target = x0[:N]

def oracle(x, t):
    ab = _extract(d.alpha_bars, t, x)
    return (x - ab.sqrt() * target) / (1 - ab).sqrt()

out = d.sample(oracle, N, device)
assert (out - target).abs().max().item() < 5e-3
print("sampling (model oracle) OK")
print("diffusion OK")